# **Soliton Solutions to the Korteweg–De Vries (KdV) equation**

Here we present a brief example of how the code is used to solve the KdV equation for one- and two-soliton solutions using physics-informed neural networks.

**Setup:**

In [1]:
import torch
import torch.nn as nn

import sys
from pathlib import Path

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from models import KDV

Check GPU availability:

In [2]:
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")

CUDA available: True


## **One soliton**

Configure a model for a single soliton:

In [3]:
INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = 72, 
    verbose                  = True,
)

TRAIN_PARAMS = dict(
    adam_epochs              = 1000,
    verbose_step             = 100,
    n_collocation            = 50000, 
    n_initial                = 30000,  
    n_boundary               = 10000,
    n_momentum               = 0, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
    n_energy                 = 0,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
    adam_lr                  = 1e-3,   
    lbfgs_lr                 = 1.0,    
    lbfgs_history_size       = 100, 
    lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
    adaptive_sampling        = False,   
    logging                  = True, #new parameter, stops loss logging bottleneck for quick training (no loss history)
    verbose                  = True,
)

TRAIN_WEIGHTS = dict( #seperated out from the train params
    w_ic                     = 5.0,    
    w_bc                     = 2.0,    
    w_pde                    = 15.0,
    w_momentum               = 1.0,
    w_energy                 = 0.1,
)

Create an instance of the class:

In [4]:
model = KDV(INIT_PARAMS)

Using device: cuda


Train the model:

In [5]:
training_stats, domain = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)

ValueError: Expected parameter_and_buffer_dicts to be a dict, or a list/tuple of dicts, but got <class 'generator'>

Let's plot how training progressed:

In [ ]:
fig = model.plot_losses(training_stats=training_stats, 
                        components=['total', 'pde', 'boundary',
                                    'initial', 'momentum','energy'])

Visualize the results:

In [ ]:
fig = model.plot_profiles(t_values=[-15, 0, 15], which=("predicted", "exact"))

In [ ]:
fig_1 = model.plot_spacetime()
fig_2 = model.plot_spacetime(scatter_which=['initial', 'pde', 
                                            'boundary'],
                             training_domain=domain) #call to scatter


We can also use the test method to view the point-wise error over the domain and also compute the mean and max error. Multiple error types are available, including `absolute` and `absolute-normalized`.

In [ ]:
fig = model.plot_heatmap()

## **Two solitons**

Configure and train for two colliding solitons:

In [ ]:
INIT_PARAMS_2 = dict(
            num_solitons=2,
            n_hidden_layers=7, 
            n_neurons_per_layer=62, 
            activation=nn.Tanh,
            seed=72, 
            verbose=True,
        )
        
TRAIN_PARAMS_2 = dict(
    adam_epochs              = 1000,
    verbose_step             = 100,
    n_collocation            = 100000, 
    n_initial                = 30000,  
    n_boundary               = 10000,
    n_momentum               = 40, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
    n_energy                 = 40,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
    adam_lr                  = 1e-3,   
    lbfgs_lr                 = 2.0,    
    lbfgs_history_size       = 100, 
    lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
    adaptive_sampling        = False,   
    logging                  = True, #new parameter, stops loss logging bottleneck for quick training (no loss history)
    verbose                  = True,
)

TRAIN_WEIGHTS_2 = dict( #seperated out from the train params
    w_ic                     = 10.0,    
    w_bc                     = 1.0,    
    w_pde                    = 100.0,
    w_momentum               = 0.005,
    w_energy                 = 0.005,
)

model_2 = KDV(INIT_PARAMS_2)
training_stats_2, domain_2 = model_2.fit(TRAIN_PARAMS_2, TRAIN_WEIGHTS_2)